# Executar produtor Kafka ()
Tech Challenge Fase 2 - Pipeline Híbrida de Alfabetização

Este notebook instala a dependência, lê as credenciais do Confluent Cloud via **Databricks Secrets** e executa o script .

**Antes de rodar:** faça upload de  para o mesmo diretório deste notebook no Workspace (ou para o Volume/DBFS de sua preferência) e ajuste o widget  na célula 3 se necessário.

## 1. Instalar a biblioteca `confluent-kafka`

In [0]:
%pip install confluent-kafka --break-system-packages

In [0]:
# Reinicia o interpretador Python para que a biblioteca recém-instalada
# fique disponível na sessão atual
dbutils.library.restartPython()

## 2. Configurar caminho do script e parâmetros
Preencha os widgets abaixo (Ícone de controles no topo do notebook).

As credenciais do Confluent Cloud são lidas automaticamente do **Databricks Secrets** (scope ).

In [0]:
dbutils.widgets.text("script_path", "producer_alunos.py", "Caminho do script")
dbutils.widgets.text("topic", "alunos-eventos", "Kafka Topic")
dbutils.widgets.text("intervalo", "2", "Intervalo (s)")
dbutils.widgets.text("quantidade", "50", "Quantidade de eventos")

In [0]:
import os

SECRET_SCOPE = "tc_02"

os.environ["CONFLUENT_BOOTSTRAP_SERVERS"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="bootstrap_servers")
os.environ["CONFLUENT_API_KEY"]            = dbutils.secrets.get(scope=SECRET_SCOPE, key="api_key")
os.environ["CONFLUENT_API_SECRET"]         = dbutils.secrets.get(scope=SECRET_SCOPE, key="api_secret")
os.environ["KAFKA_TOPIC"]                  = dbutils.widgets.get("topic")

script_path = dbutils.widgets.get("script_path")
if not os.path.exists(script_path):
    raise FileNotFoundError(
        f"Script não encontrado em '{script_path}'. "
        "Confira o widget 'script_path' e se o arquivo foi enviado ao workspace."
    )

print("Credenciais carregadas do scope:", SECRET_SCOPE)
print("Tópico:", os.environ["KAFKA_TOPIC"])
print("Script localizado em:", script_path)

## 3. Executar o produtor
Equivalente a rodar `python producer_alunos.py --intervalo 2 --quantidade 50` no terminal, mas passando o ambiente (`os.environ`) explicitamente para o subprocesso — assim as credenciais chegam ao script mesmo sem `export` persistir entre células.

In [0]:
import subprocess

comando = [
    "python", script_path,
    "--intervalo", dbutils.widgets.get("intervalo"),
    "--quantidade", dbutils.widgets.get("quantidade"),
]

processo = subprocess.Popen(
    comando,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for linha in processo.stdout:
    print(linha, end="")

processo.wait()
print(f"\nProcesso finalizado com código de saída {processo.returncode}")